# 🛣️ Multi-Prompt Road Damage & Infrastructure Segmentation
### Menggunakan SAM (Segment Anything Model 2.1 / SAM 3.1) & Open-Vocabulary VLM Prompting

Notebook ini dirancang khusus untuk penelitian **Hibah VLM (Vision-Language Model)** dalam mendeteksi dan melakukan segmentasi presisi tinggi pada berbagai kerusakan jalan serta infrastruktur lalu lintas (**Pothole, Manhole, Traffic Sign, Road Crack**, dll.) menggunakan paradigma **Prompt-Driven Segmentation**.

---

### 🌟 Fitur Utama Notebook Ini:
1. **Arsitektur SAM 2.1 & SAM 3.1 Terkini**:
   - Mendukung **SAM 2.1** (`sam2.1_t.pt` / `sam2.1_s.pt`) yang sangat efisien pada GPU lokal (RTX 3050 4GB VRAM) dengan akselerasi CUDA.
   - Dilengkapi sistem **Auto-Switch ke SAM 3** jika bobot resmi `sam3.pt` diletakkan di direktori kerja.
2. **Open-Vocabulary Vision-Language Grounding**:
   - Menggunakan model prompt bahasa alami (YOLO-World + CLIP) yang dapat memahami teks bebas (*zero-shot prompting*) tanpa perlu melatih ulang model.
3. **Pengujian Prompt Terpisah (Individual Testing)**:
   - Pengujian mandiri sel demi sel untuk prompt spesifik: **`pothole`**, **`manhole`**, **`sign`**, dan **`crack`**.
4. **Unified Multi-Prompt Pipeline (Penggabungan Seluruh Prompt)**:
   - Eksekusi simultan seluruh prompt dalam satu pipeline lengkap dengan pewarnaan unik per kelas, kontur mask poligon SAM, dan ringkasan metrik keparahan (*damage severity index*).
5. **Inferensi Video (`testvideo1.mp4`) & Placeholder Kelas Dinamis**:
   - Dilengkapi **Placeholder Konfigurasi Target** yang fleksibel: Anda bebas menambah, mengurangi, atau mengganti kelas prompt apa saja (misal menambah kelas mobil, pejalan kaki, marka jalan, dll.).
   - Menghasilkan video beresolusi tinggi dengan overlay mask transparan, bounding box, dan HUD telemetry real-time (FPS & counter deteksi).

## 1. Setup Environment & Inisialisasi Dependensi
Memeriksa kesiapan PyTorch, akselerasi GPU CUDA (NVIDIA RTX 3050), pustaka Ultralytics, serta OpenCV.

In [ ]:
import os
import cv2
import torch
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import ultralytics
from ultralytics import YOLO, SAM

print(f"Ultralytics version : {ultralytics.__version__}")
print(f"PyTorch version     : {torch.__version__}")
print(f"CUDA Available       : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device Name      : {torch.cuda.get_device_name(0)}")
    print(f"VRAM Total           : {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
    device_name = 0
else:
    print("⚠️ CUDA tidak terdeteksi, inferensi berjalan di CPU.")
    device_name = "cpu"


## 2. Inisialisasi Model: SAM 2.1 / SAM 3.1 & Open-Vocabulary Prompt Engine

> [!NOTE]
> **Klarifikasi Checkpoint:**
> - Meta merilis SAM 2.1 (arsitektur video streaming & mask decoder terbaru) serta SAM 3 / 3.1 (Promptable Concept Segmentation berbasis teks).
> - Bobot `sam2.1_t.pt` (SAM 2.1 Tiny) sangat ringan, berkecepatan tinggi, dan hemat memori (~1.5 GB VRAM), sangat ideal untuk GPU 4GB RTX 3050.
> - Pipeline di bawah memadukan **Open-Vocabulary Language Grounding (YOLO-World v2 + CLIP)** sebagai penerjemah prompt teks ke koordinat spasial, yang langsung diteruskan ke **SAM 2.1** untuk menghasilkan segmentasi mask poligon berpresisi tinggi.

In [ ]:
# 1. Pemuatan Model SAM (Adaptif: SAM 3 jika sam3.pt ada di folder, atau SAM 2.1)
sam_checkpoint = "sam3.pt" if Path("sam3.pt").exists() else "sam2.1_t.pt"
print(f"Loading SAM model from: {sam_checkpoint}...")
sam_model = SAM(sam_checkpoint)

# 2. Pemuatan Model Open-Vocabulary Prompt Grounding (YOLO-World)
# Mampu mendeteksi konsep teks apa saja via prompt bahasa alami
PROMPT_GROUNDING_MODEL = "yolov8s-worldv2.pt"
print(f"Loading Open-Vocabulary Grounding Model: {PROMPT_GROUNDING_MODEL}...")
prompt_grounding_net = YOLO(PROMPT_GROUNDING_MODEL)

# 3. Pemuatan Model Deteksi Fine-Tuned (Opsional / Benchmark Kerusakan Jalan)
custom_weights = Path("runs/segment/runs/segment/pothole_seg_v1/weights/best.pt")
if custom_weights.exists():
    print(f"Loading Custom Fine-Tuned Weights: {custom_weights}...")
    custom_trained_net = YOLO(str(custom_weights))
else:
    custom_trained_net = None

print("✅ Seluruh model berhasil diinisialisasi!")


## 3. Fungsi Inti Pipeline Prompt ➡️ SAM Segmentation

Fungsi `run_prompt_sam_segmentation` mengeksekusi alur dua tahap:
1. **Language-to-Object Grounding**: Mengubah prompt teks menjadi bounding box kandidat lokasi objek.
2. **SAM High-Resolution Mask Generation**: Mengirim bounding box ke SAM sebagai prompt spasial untuk menghasilkan mask poligon beresolusi penuh yang presisi mengikuti tepi kerusakan jalan.

In [ ]:
def run_prompt_sam_segmentation(
    image_source,
    target_prompts,
    prompt_net,
    sam_net,
    conf=0.15,
    iou=0.5,
    imgsz=800,
    class_colors=None,
    device=0
):
    """
    Menjalankan segmentasi berbasis prompt:
    - image_source: path gambar (str/Path) atau array numpy BGR (frame video/gambar)
    - target_prompts: list teks prompt, contoh ['pothole', 'manhole', 'traffic sign', 'road crack']
    - prompt_net: model YOLO-World open-vocabulary
    - sam_net: model SAM 2.1 / SAM 3
    """
    if isinstance(image_source, (str, Path)):
        img = cv2.imread(str(image_source))
        if img is None:
            raise IOError(f"Gambar tidak ditemukan: {image_source}")
    else:
        img = image_source.copy()

    h, w = img.shape[:2]
    
    # Standarisasi target prompt ke format list
    if isinstance(target_prompts, str):
        prompts_list = [target_prompts]
    else:
        prompts_list = list(target_prompts)

    # Set vocabulary kelas baru pada model grounding secara real-time
    prompt_net.set_classes(prompts_list)

    # Step 1: Deteksi Bounding Box berbasis teks prompt
    det_results = prompt_net.predict(
        img, 
        conf=conf, 
        iou=iou, 
        imgsz=imgsz, 
        verbose=False,
        device=device
    )[0]

    overlay = img.copy()
    detection_records = []

    # Jika tidak ada deteksi, kembalikan gambar asli
    if len(det_results.boxes) == 0:
        return overlay, detection_records, None, det_results

    boxes_xyxy = det_results.boxes.xyxy.cpu().numpy()
    cls_ids = det_results.boxes.cls.cpu().numpy().astype(int)
    confs = det_results.boxes.conf.cpu().numpy()

    # Palet warna default jika class_colors tidak disediakan
    default_colors = [
        (0, 0, 255),    # Merah
        (0, 255, 0),    # Hijau
        (0, 255, 255),  # Kuning
        (255, 0, 0),    # Biru
        (255, 0, 255),  # Magenta
        (0, 165, 255),  # Oranye
    ]

    # Step 2: Kirim Bounding Box ke SAM sebagai Box Prompt
    sam_results = sam_net.predict(
        source=img,
        bboxes=boxes_xyxy.tolist(),
        retina_masks=True,
        verbose=False,
        device=device
    )[0]

    if sam_results.masks is not None:
        masks_np = sam_results.masks.data.cpu().numpy()  # [N, H, W]
        for i in range(len(masks_np)):
            mask_i = (masks_np[i] > 0.5).astype(np.uint8)
            if mask_i.shape != (h, w):
                mask_i = cv2.resize(mask_i, (w, h), interpolation=cv2.INTER_NEAREST)

            cls_id = cls_ids[i] if i < len(cls_ids) else 0
            prompt_name = prompts_list[cls_id] if cls_id < len(prompts_list) else f"cls_{cls_id}"
            conf_val = float(confs[i]) if i < len(confs) else 1.0

            # Penentuan warna kelas
            if class_colors and prompt_name in class_colors:
                color = class_colors[prompt_name]
            else:
                color = default_colors[cls_id % len(default_colors)]

            # 1. Overlay Mask Transparan
            colored_mask = np.zeros_like(img, dtype=np.uint8)
            colored_mask[mask_i == 1] = color
            overlay = cv2.addWeighted(overlay, 1.0, colored_mask, 0.45, 0)

            # 2. Gambar Garis Kontur Halus
            contours, _ = cv2.findContours(mask_i, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            cv2.drawContours(overlay, contours, -1, color, 2)

            # 3. Bounding Box & Label Teks
            x1, y1, x2, y2 = map(int, boxes_xyxy[i])
            cv2.rectangle(overlay, (x1, y1), (x2, y2), color, 2)

            label_text = f"{prompt_name} {conf_val:.2f}"
            (tw, th), _ = cv2.getTextSize(label_text, cv2.FONT_HERSHEY_SIMPLEX, 0.55, 2)
            cv2.rectangle(overlay, (x1, max(0, y1 - th - 8)), (x1 + tw + 6, y1), color, -1)
            cv2.putText(overlay, label_text, (x1 + 3, y1 - 4), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 2)

            # Hitung statistik luas area
            pixel_area = int(np.sum(mask_i))
            area_ratio = (pixel_area / (h * w)) * 100.0

            detection_records.append({
                "prompt": prompt_name,
                "confidence": conf_val,
                "box": [x1, y1, x2, y2],
                "pixel_area": pixel_area,
                "area_percent": area_ratio
            })

    return overlay, detection_records, sam_results, det_results

print("✅ Fungsi 'run_prompt_sam_segmentation' siap digunakan!")


## 4. Pengujian Prompt Terpisah (Individual Prompt Testing)

Sesuai kebutuhan riset, di bagian ini kita menguji masing-masing teks prompt secara **terpisah** untuk melihat ketepatan deteksi dan kualitas mask SAM pada masing-masing target:
- **Prompt A:** `pothole` (Lubang Jalan)
- **Prompt B:** `manhole` (Tutup Gorong-gorong/Got)
- **Prompt C:** `sign` / `traffic sign` (Rambu Jalan)
- **Prompt D:** `crack` / `road crack` (Retakan Aspal)

In [ ]:
# ==============================================================================
# 🕳️ PENGUJIAN PROMPT TERPISAH 1: POTHOLE (LUBANG JALAN)
# ==============================================================================
TEST_IMG = "testimage3.png"  # Gambar sampel jalan berlubang dan retak
PROMPT_1 = "pothole"

print(f"Menjalankan segmentasi terpisah untuk prompt: '{PROMPT_1}'...")
res_img_1, dets_1, sam_res_1, _ = run_prompt_sam_segmentation(
    image_source=TEST_IMG,
    target_prompts=[PROMPT_1],
    prompt_net=prompt_grounding_net,
    sam_net=sam_model,
    conf=0.15,
    class_colors={PROMPT_1: (0, 0, 255)}  # Merah
)

# Visualisasi Side-by-Side
img_raw = cv2.cvtColor(cv2.imread(TEST_IMG), cv2.COLOR_BGR2RGB)
res_rgb_1 = cv2.cvtColor(res_img_1, cv2.COLOR_BGR2RGB)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].imshow(img_raw)
axes[0].set_title(f"Gambar Asli: {TEST_IMG}", fontsize=12)
axes[0].axis("off")

axes[1].imshow(res_rgb_1)
axes[1].set_title(f"Hasil Segmentasi Prompt: '{PROMPT_1}' ({len(dets_1)} terdeteksi)", fontsize=12)
axes[1].axis("off")
plt.tight_layout()
plt.show()

print(f"📊 Ringkasan Deteksi [{PROMPT_1}]:")
for idx, d in enumerate(dets_1, 1):
    print(f"  {idx}. Conf: {d['confidence']:.2f} | Luas: {d['pixel_area']:,} px ({d['area_percent']:.2f}%)")


In [ ]:
# ==============================================================================
# 🕳️ PENGUJIAN PROMPT TERPISAH 2: MANHOLE (TUTUP GORONG-GORONG / SALURAN AIR)
# ==============================================================================
TEST_IMG = "testimage3.png"
PROMPT_2 = "manhole"

print(f"Menjalankan segmentasi terpisah untuk prompt: '{PROMPT_2}'...")
res_img_2, dets_2, sam_res_2, _ = run_prompt_sam_segmentation(
    image_source=TEST_IMG,
    target_prompts=[PROMPT_2],
    prompt_net=prompt_grounding_net,
    sam_net=sam_model,
    conf=0.15,
    class_colors={PROMPT_2: (0, 255, 0)}  # Hijau
)

# Visualisasi Side-by-Side
res_rgb_2 = cv2.cvtColor(res_img_2, cv2.COLOR_BGR2RGB)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].imshow(img_raw)
axes[0].set_title(f"Gambar Asli: {TEST_IMG}", fontsize=12)
axes[0].axis("off")

axes[1].imshow(res_rgb_2)
axes[1].set_title(f"Hasil Segmentasi Prompt: '{PROMPT_2}' ({len(dets_2)} terdeteksi)", fontsize=12)
axes[1].axis("off")
plt.tight_layout()
plt.show()

print(f"📊 Ringkasan Deteksi [{PROMPT_2}]:")
for idx, d in enumerate(dets_2, 1):
    print(f"  {idx}. Conf: {d['confidence']:.2f} | Luas: {d['pixel_area']:,} px ({d['area_percent']:.2f}%)")


In [ ]:
# ==============================================================================
# 🛑 PENGUJIAN PROMPT TERPISAH 3: SIGN / TRAFFIC SIGN (RAMBU LALU LINTAS)
# ==============================================================================
TEST_IMG_SIGN = "testimage4.png" if Path("testimage4.png").exists() else "testimage3.png"
PROMPT_3 = "traffic sign"

print(f"Menjalankan segmentasi terpisah untuk prompt: '{PROMPT_3}' pada {TEST_IMG_SIGN}...")
res_img_3, dets_3, sam_res_3, _ = run_prompt_sam_segmentation(
    image_source=TEST_IMG_SIGN,
    target_prompts=[PROMPT_3],
    prompt_net=prompt_grounding_net,
    sam_net=sam_model,
    conf=0.15,
    class_colors={PROMPT_3: (0, 255, 255)}  # Kuning
)

# Visualisasi Side-by-Side
img_raw_sign = cv2.cvtColor(cv2.imread(TEST_IMG_SIGN), cv2.COLOR_BGR2RGB)
res_rgb_3 = cv2.cvtColor(res_img_3, cv2.COLOR_BGR2RGB)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].imshow(img_raw_sign)
axes[0].set_title(f"Gambar Asli: {TEST_IMG_SIGN}", fontsize=12)
axes[0].axis("off")

axes[1].imshow(res_rgb_3)
axes[1].set_title(f"Hasil Segmentasi Prompt: '{PROMPT_3}' ({len(dets_3)} terdeteksi)", fontsize=12)
axes[1].axis("off")
plt.tight_layout()
plt.show()

print(f"📊 Ringkasan Deteksi [{PROMPT_3}]:")
for idx, d in enumerate(dets_3, 1):
    print(f"  {idx}. Conf: {d['confidence']:.2f} | Luas: {d['pixel_area']:,} px ({d['area_percent']:.2f}%)")


In [ ]:
# ==============================================================================
# ⚡ PENGUJIAN PROMPT TERPISAH 4: ROAD CRACK (RETAKAN JALAN)
# ==============================================================================
TEST_IMG = "testimage3.png"
PROMPT_4 = "road crack"

print(f"Menjalankan segmentasi terpisah untuk prompt: '{PROMPT_4}'...")
res_img_4, dets_4, sam_res_4, _ = run_prompt_sam_segmentation(
    image_source=TEST_IMG,
    target_prompts=[PROMPT_4],
    prompt_net=prompt_grounding_net,
    sam_net=sam_model,
    conf=0.15,
    class_colors={PROMPT_4: (255, 0, 0)}  # Biru
)

# Visualisasi Side-by-Side
res_rgb_4 = cv2.cvtColor(res_img_4, cv2.COLOR_BGR2RGB)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].imshow(img_raw)
axes[0].set_title(f"Gambar Asli: {TEST_IMG}", fontsize=12)
axes[0].axis("off")

axes[1].imshow(res_rgb_4)
axes[1].set_title(f"Hasil Segmentasi Prompt: '{PROMPT_4}' ({len(dets_4)} terdeteksi)", fontsize=12)
axes[1].axis("off")
plt.tight_layout()
plt.show()

print(f"📊 Ringkasan Deteksi [{PROMPT_4}]:")
for idx, d in enumerate(dets_4, 1):
    print(f"  {idx}. Conf: {d['confidence']:.2f} | Luas: {d['pixel_area']:,} px ({d['area_percent']:.2f}%)")


## 5. Unified Multi-Prompt Pipeline (Menggabungkan Seluruh Prompt Menjadi Satu)

Setelah menguji masing-masing prompt secara terpisah, kini seluruh target prompt (**pothole, manhole, traffic sign, road crack**) dieksekusi secara **simultan** dalam satu pipeline komprehensif.

### 🎨 Sistem Pewarnaan Terstandarisasi:
- **Pothole (Lubang Jalan)**: 🔴 Merah `(0, 0, 255)`
- **Road Crack (Retakan Aspal)**: 🔵 Biru `(255, 0, 0)`
- **Manhole (Tutup Gorong-gorong)**: 🟢 Hijau `(0, 255, 0)`
- **Traffic Sign (Rambu Jalan)**: 🟡 Kuning `(0, 255, 255)`

In [ ]:
# Daftar seluruh target prompt gabungan
UNIFIED_PROMPTS = [
    "pothole", 
    "road crack", 
    "manhole", 
    "traffic sign"
]

UNIFIED_COLORS = {
    "pothole": (0, 0, 255),       # Merah
    "road crack": (255, 0, 0),    # Biru
    "manhole": (0, 255, 0),       # Hijau
    "traffic sign": (0, 255, 255) # Kuning
}

IMAGE_SAMPLE = "testimage3.png"
print(f"Menjalankan Unified Multi-Prompt Pipeline pada: {IMAGE_SAMPLE}...")

unified_overlay, all_dets, sam_out, yolo_out = run_prompt_sam_segmentation(
    image_source=IMAGE_SAMPLE,
    target_prompts=UNIFIED_PROMPTS,
    prompt_net=prompt_grounding_net,
    sam_net=sam_model,
    conf=0.15,
    class_colors=UNIFIED_COLORS
)

# Visualisasi Komparasi 3-Panel
img_original = cv2.cvtColor(cv2.imread(IMAGE_SAMPLE), cv2.COLOR_BGR2RGB)
boxes_only_plot = cv2.cvtColor(yolo_out.plot(), cv2.COLOR_BGR2RGB)
unified_plot = cv2.cvtColor(unified_overlay, cv2.COLOR_BGR2RGB)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
axes[0].imshow(img_original)
axes[0].set_title("1. Gambar Asli Permukaan Jalan", fontsize=12)
axes[0].axis("off")

axes[1].imshow(boxes_only_plot)
axes[1].set_title("2. Deteksi Bounding Box Teks Prompt", fontsize=12)
axes[1].axis("off")

axes[2].imshow(unified_plot)
axes[2].set_title("3. Hasil Segmentasi Presisi Tinggi SAM 2.1", fontsize=12)
axes[2].axis("off")

plt.tight_layout()
plt.show()

# Simpan hasil gabungan
cv2.imwrite("hasil_unified_prompt_sam.jpg", unified_overlay)
print("✅ Hasil segmentasi gabungan disimpan ke: 'hasil_unified_prompt_sam.jpg'")

# ==============================================================================
# 📊 TABEL REKAPITULASI METRIK KERUSAKAN JALAN (VLM DAMAGE ASSESSMENT)
# ==============================================================================
total_img_pixels = img_original.shape[0] * img_original.shape[1]
damage_summary = {}

for d in all_dets:
    p = d["prompt"]
    if p not in damage_summary:
        damage_summary[p] = {"count": 0, "total_pixels": 0}
    damage_summary[p]["count"] += 1
    damage_summary[p]["total_pixels"] += d["pixel_area"]

print("\n" + "=" * 65)
print("📑 LAPORAN METRIK SEGMENTASI PROMPT GABUNGAN (HIBAH VLM)")
print("=" * 65)
print(f"{'Target Prompt':<18} | {'Jumlah Objek':<14} | {'Total Luas (px)':<16} | {'Rasio Jalan (%)'}")
print("-" * 65)
for prompt_name, stats in damage_summary.items():
    pct = (stats["total_pixels"] / total_img_pixels) * 100.0
    print(f"{prompt_name:<18} | {stats['count']:<14} | {stats['total_pixels']:<16,} | {pct:.2f}%")
print("=" * 65)


## 6. Bagian 2: Inferensi Video (`testvideo1.mp4`) dengan Placeholder Kelas Dinamis

Di bagian ini, sistem inferensi diterapkan langsung pada rekaman video jalan (**`testvideo1.mp4`**).

### 🎯 Desain Placeholder Target Fleksibel:
Anda dapat **menambah, menghapus, atau mengganti** kelas prompt apa saja pada dictionary `VIDEO_PROMPT_CONFIG` di bawah ini. Pipeline akan secara otomatis mengadaptasi prompt baru tanpa perlu pelatihan ulang!

In [ ]:
# ==============================================================================
# 🎯 PLACEHOLDER KONFIGURASI TARGET DETEKSI & SEGMENTASI VIDEO
# Anda bebas menambah, mengurangi, atau mengganti kelas prompt apa saja di sini!
# Format:
#   "nama_prompt": {
#       "label": "Nama Tampilan pada Layar",
#       "color": (B, G, R),     # Kode Warna BGR OpenCV
#       "conf": threshold_nilai # Nilai ambang keyakinan (0.1 - 0.9)
#   }
# ==============================================================================
VIDEO_PROMPT_CONFIG = {
    "pothole": {
        "label": "Pothole (Lubang)",
        "color": (0, 0, 255),      # Merah
        "conf": 0.15
    },
    "road crack": {
        "label": "Crack (Retakan)",
        "color": (255, 0, 0),      # Biru
        "conf": 0.15
    },
    "manhole": {
        "label": "Manhole (Tutup Got)",
        "color": (0, 255, 0),      # Hijau
        "conf": 0.18
    },
    "traffic sign": {
        "label": "Sign (Rambu Jalan)",
        "color": (0, 255, 255),    # Kuning
        "conf": 0.20
    },

    # 💡 CONTOH CARA MENAMBAH KELAS BARU (Cukup hilangkan tanda komentar '#' di bawah):
    # "car": {
    #     "label": "Mobil/Kendaraan",
    #     "color": (255, 0, 255),  # Magenta
    #     "conf": 0.25
    # },
    # "pedestrian": {
    #     "label": "Pejalan Kaki",
    #     "color": (255, 255, 0),  # Cyan
    #     "conf": 0.25
    # },
    # "crosswalk": {
    #     "label": "Zebra Cross",
    #     "color": (180, 255, 0),  # Lime
    #     "conf": 0.20
    # },
}

print("✅ Konfigurasi Placeholder Video aktif:")
for k, v in VIDEO_PROMPT_CONFIG.items():
    print(f"  • Prompt: '{k}' -> Label: '{v['label']}', Ambang Conf: {v['conf']}")


In [ ]:
import time

def process_road_video_prompt_sam(
    video_path,
    output_path,
    prompt_config,
    prompt_net,
    sam_net,
    max_frames=90,           # Set ke None jika ingin memproses video penuh
    skip_frames=1,           # 1 = proses setiap frame, 2 = setiap 2 frame (lebih cepat)
    resize_dim=(1280, 720)   # Resolusi pemrosesan untuk efisiensi VRAM & kecepatan FPS
):
    """
    Memproses video uji menggunakan YOLO-World Open-Vocabulary Grounding + SAM 2.1 Refiner.
    Dilengkapi HUD telemetry real-time: FPS counter, jumlah deteksi per kelas, dan mask transparan.
    """
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise IOError(f"Video tidak ditemukan: {video_path}")

    orig_w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    orig_h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps_in = cap.get(cv2.CAP_PROP_FPS) or 30.0
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    out_w, out_h = resize_dim if resize_dim else (orig_w, orig_h)
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    writer = cv2.VideoWriter(str(output_path), fourcc, fps_in / skip_frames, (out_w, out_h))

    # Ekstrak daftar prompt teks dan warna dari placeholder
    target_prompts = list(prompt_config.keys())
    class_colors = {k: v["color"] for k, v in prompt_config.items()}
    min_conf = min(v["conf"] for v in prompt_config.values())

    print("=" * 60)
    print(f"🎬 MEMULAI PEMROSESAN VIDEO: {video_path}")
    print(f"Resolusi Input : {orig_w}x{orig_h} @ {fps_in:.1f} FPS")
    print(f"Resolusi Output: {out_w}x{out_h}")
    print(f"Target Prompts : {target_prompts}")
    print(f"Batas Frame    : {max_frames if max_frames else 'Seluruh Frame (' + str(total_frames) + ')'}")
    print("=" * 60)

    frame_idx = 0
    processed_count = 0
    start_time = time.time()
    
    temporal_damage_log = []
    sample_frames_display = []

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        if frame_idx % skip_frames != 0:
            frame_idx += 1
            continue

        if max_frames and processed_count >= max_frames:
            break

        t_frame_start = time.time()

        if resize_dim:
            frame_resized = cv2.resize(frame, resize_dim)
        else:
            frame_resized = frame

        # Jalankan segmentasi prompt SAM
        annotated_frame, frame_dets, _, _ = run_prompt_sam_segmentation(
            image_source=frame_resized,
            target_prompts=target_prompts,
            prompt_net=prompt_net,
            sam_net=sam_net,
            conf=min_conf,
            class_colors=class_colors
        )

        t_frame_end = time.time()
        curr_fps = 1.0 / max(1e-5, (t_frame_end - t_frame_start))

        # ======================================================================
        # RENDER HUD DASHBOARD OVERLAY PADA VIDEO
        # ======================================================================
        # Header Box semi-transparan
        hud_bg = annotated_frame.copy()
        cv2.rectangle(hud_bg, (10, 10), (430, 75 + len(target_prompts) * 22), (20, 20, 20), -1)
        annotated_frame = cv2.addWeighted(annotated_frame, 0.25, hud_bg, 0.75, 0)

        # Informasi Telemetry
        cv2.putText(annotated_frame, f"SAM 2.1 Road Inspection | Frame: {frame_idx}/{total_frames}", 
                    (20, 32), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 2)
        cv2.putText(annotated_frame, f"Kecepatan: {curr_fps:.1f} FPS | Durasi: {(frame_idx / fps_in):.2f}s", 
                    (20, 54), cv2.FONT_HERSHEY_SIMPLEX, 0.50, (0, 255, 200), 1)

        # Ringkasan Deteksi Per Kelas di Layar
        counts_per_class = {p: 0 for p in target_prompts}
        total_frame_dmg_px = 0
        for d in frame_dets:
            counts_per_class[d["prompt"]] = counts_per_class.get(d["prompt"], 0) + 1
            total_frame_dmg_px += d["pixel_area"]

        y_offset = 78
        for p in target_prompts:
            col = class_colors.get(p, (255, 255, 255))
            disp_label = prompt_config[p]["label"]
            cnt = counts_per_class.get(p, 0)
            cv2.circle(annotated_frame, (28, y_offset - 4), 6, col, -1)
            cv2.putText(annotated_frame, f"{disp_label}: {cnt}", 
                        (42, y_offset), cv2.FONT_HERSHEY_SIMPLEX, 0.48, (255, 255, 255), 1)
            y_offset += 22

        writer.write(annotated_frame)

        # Simpan log metrik temporal
        temporal_damage_log.append({
            "frame": frame_idx,
            "time_sec": frame_idx / fps_in,
            "counts": counts_per_class,
            "damage_pixels": total_frame_dmg_px,
            "damage_pct": (total_frame_dmg_px / (out_w * out_h)) * 100.0
        })

        # Ambil sampel frame untuk visualisasi galeri
        if processed_count in [0, max_frames // 4, max_frames // 2, (max_frames * 3) // 4, max_frames - 1]:
            sample_frames_display.append((frame_idx, annotated_frame.copy()))

        processed_count += 1
        frame_idx += 1

        if processed_count % 15 == 0 or processed_count == max_frames:
            elapsed = time.time() - start_time
            print(f"  ⚡ Diproses: {processed_count}/{max_frames or total_frames} frames | Kecepatan rata-rata: {processed_count/elapsed:.1f} FPS")

    cap.release()
    writer.release()

    total_time = time.time() - start_time
    print("=" * 60)
    print(f"✅ PEMROSESAN VIDEO SELESAI!")
    print(f"Total Frame Diproses: {processed_count} frames")
    print(f"Total Waktu          : {total_time:.2f} detik ({processed_count/total_time:.1f} FPS rata-rata)")
    print(f"File Hasil Disimpan  : {output_path}")
    print("=" * 60)

    return temporal_damage_log, sample_frames_display


In [ ]:
# ==============================================================================
# 🚀 EKSEKUSI INFERENSI VIDEO PADA 'testvideo1.mp4'
# ==============================================================================
INPUT_VIDEO = "testvideo1.mp4"
OUTPUT_VIDEO = "hasil_video_prompt_sam.mp4"

# Catatan: Kita atur max_frames=90 (sekitar 3 detik cuplikan) untuk verifikasi cepat.
# Ubah max_frames=None untuk memproses keseluruhan video.
damage_log, sample_frames = process_road_video_prompt_sam(
    video_path=INPUT_VIDEO,
    output_path=OUTPUT_VIDEO,
    prompt_config=VIDEO_PROMPT_CONFIG,
    prompt_net=prompt_grounding_net,
    sam_net=sam_model,
    max_frames=90,           # Ganti ke None untuk full video
    skip_frames=1,
    resize_dim=(1280, 720)
)

# Visualisasi Galeri Sampel Frame Hasil Video
if sample_frames:
    num_samples = len(sample_frames)
    fig, axes = plt.subplots(1, num_samples, figsize=(5 * num_samples, 4))
    if num_samples == 1:
        axes = [axes]
        
    for ax, (f_idx, f_img) in zip(axes, sample_frames):
        ax.imshow(cv2.cvtColor(f_img, cv2.COLOR_BGR2RGB))
        ax.set_title(f"Frame ke-{f_idx}", fontsize=11)
        ax.axis("off")
        
    plt.suptitle("📸 Galeri Cuplikan Inferensi Video (Prompt + SAM 2.1)", fontsize=14, y=1.03)
    plt.tight_layout()
    plt.show()


## 7. Analisis Metrik Temporal Kerusakan Jalan (Hibah VLM Report)

Grafik fluktuasi area kerusakan jalan sepanjang durasi video untuk mengidentifikasi titik jalan dengan tingkat keparahan tertinggi (*critical road segments*).

In [ ]:
if damage_log:
    times = [d["time_sec"] for d in damage_log]
    dmg_pcts = [d["damage_pct"] for d in damage_log]
    pothole_counts = [d["counts"].get("pothole", 0) for d in damage_log]
    crack_counts = [d["counts"].get("road crack", 0) for d in damage_log]

    fig, ax1 = plt.subplots(figsize=(14, 5))

    color_dmg = 'tab:red'
    ax1.set_xlabel("Waktu Video (detik)", fontsize=11)
    ax1.set_ylabel("Rasio Luas Kerusakan Jalan (%)", color=color_dmg, fontsize=11)
    line1 = ax1.plot(times, dmg_pcts, color=color_dmg, linewidth=2, label="Rasio Luas Kerusakan (%)")
    ax1.tick_params(axis='y', labelcolor=color_dmg)
    ax1.grid(True, linestyle="--", alpha=0.5)

    ax2 = ax1.twinx()
    color_cnt = 'tab:blue'
    ax2.set_ylabel("Jumlah Deteksi Objek", color=color_cnt, fontsize=11)
    line2 = ax2.plot(times, pothole_counts, color="red", linestyle=":", label="Jumlah Pothole")
    line3 = ax2.plot(times, crack_counts, color="blue", linestyle="--", label="Jumlah Retakan (Crack)")
    ax2.tick_params(axis='y', labelcolor=color_cnt)

    lines = line1 + line2 + line3
    labels = [l.get_label() for l in lines]
    ax1.legend(lines, labels, loc="upper left")

    plt.title("📈 Fluktuasi Temporal Kerusakan Permukaan Jalan Sepanjang Video (Hibah VLM)", fontsize=13)
    plt.tight_layout()
    plt.savefig("grafik_analisis_temporal_jalan.png", dpi=300)
    plt.show()

    max_idx = int(np.argmax(dmg_pcts))
    print("=" * 60)
    print("📊 KESIMPULAN ANALISIS TEMPORAL:")
    print(f"  • Puncak Kerusakan Tertinggi : Frame ke-{damage_log[max_idx]['frame']} (Detik ke-{times[max_idx]:.2f})")
    print(f"  • Persentase Kerusakan Maks  : {dmg_pcts[max_idx]:.2f}% dari luas jalan")
    print(f"  • Rekapitulasi Deteksi Maks  : {damage_log[max_idx]['counts']}")
    print("=" * 60)
